In [0]:
from pyspark.sql import functions as f
from pyspark.sql.types import *
import re
from datetime import datetime
from functools import reduce

data = [
    ('Mahesh','Mahesh','15000','16000'),
    ('Rajesh','Rajesh','25000','18000'),
    ('Mohan','MohanV','25000','25000'),
    ('Govinda','GovindaS','25000','26000'),
]

schema = StructType([
    StructField('Name_SRC',StringType()),
    StructField('Name_TGT',StringType()),
    StructField('Salary_SRC',StringType()),
    StructField('Salary_TGT',StringType()),
])

Issue = spark.createDataFrame(data=data,schema=schema)
columns = [col.replace('_SRC','') for col in Issue.columns if '_SRC' in col]


res = []
for col in columns:
    new = Issue.select(
        f.col(f"{col}_SRC").alias('left_value'),
        f.col(f"{col}_TGT").alias('right_value'),
        f.lit(col).alias('column_name'),
        f.when((f.col(f"{col}_SRC") == f.col(f"{col}_TGT")),'Matched').otherwise('Mismatch').alias('Results')
    )
    res.append(new)

final = reduce(lambda left_value,right_value:left_value.union(right_value),res)
final = final.select('column_name','left_value','right_value','Results')
final.show()
